In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from ai_ledger_guard.rules import *

df = pd.read_csv("../data/raw/journal_entries_4203.csv", sep=";")

alerts = rule_weekend(df)
print(alerts.sum())

31537


In [3]:
import duckdb

In [4]:
duckdb.sql(
    """
    SELECT
        COUNT(Weekend)
    FROM df
    WHERE Weekend = 1 OR Weekend = 2
    """
)

┌────────────────┐
│ count(Weekend) │
│     int64      │
├────────────────┤
│          31537 │
└────────────────┘

In [5]:
is_saturday = df["weekend"] == 1
is_sunday = df["weekend"] == 2
print(df[is_saturday | is_sunday]["weekend"].value_counts().sum())
print((df["weekend"] > 0).sum())

31537
31537


In [6]:
differences = alerts != (df["weekend"] > 0)
print(differences.sum())


0


In [7]:
round_alert = rule_round_amount(df)
print(round_alert.sum())

466


In [8]:
rare_acc_alert = rule_rare_account(df)
print(rare_acc_alert.sum())

286


In [9]:
rare_accounts = duckdb.sql(
    """
    SELECT
        gl_account,
        gl_account_name,
        COUNT(*) AS lines
    FROM df
    GROUP BY gl_account, gl_account_name
    HAVING COUNT(*) < 100
    ORDER BY lines
    """
).df()

print(rare_accounts)
print(rare_accounts["lines"].sum())


    gl_account                    gl_account_name  lines
0         2020              Prior-period expenses      1
1         1700                  Other liabilities      2
2          980                   Prepaid expenses      2
3         4360                 Insurance premiums      2
4         4950      Legal and consulting expenses     24
5         4240            Gas, electricity, water     24
6         4600               Advertising expenses     24
7         4210            Rent immovable property     24
8         4250                           Cleaning     24
9         4930                    Office supplies     24
10        4710                Packaging materials     24
11        4580                Other vehicle costs     24
12        4260  Maintenance of operating premises     27
13        1200                               Bank     60
286


In [10]:
rare_accounts_with_anomalies = duckdb.sql(
    """
    SELECT
        gl_account,
        gl_account_name,
        COUNT(*) AS lines,
        SUM(label) AS anomalies
    FROM df
    GROUP BY gl_account, gl_account_name
    HAVING COUNT(*) < 100 AND SUM(label) > 0
    ORDER BY lines
    """
).df()

print(rare_accounts_with_anomalies)


   gl_account          gl_account_name  lines  anomalies
0        4210  Rent immovable property     24        2.0
1        4240  Gas, electricity, water     24        1.0
2        1200                     Bank     60       60.0


In [11]:
df["weekend_alert"] = rule_weekend(df)

weekend_postings = df.groupby("posting_id")["weekend_alert"].any()

print(len(weekend_postings))
print(weekend_postings.sum())

102429
15694


In [12]:
anomaly_postings = df.groupby("posting_id")["label"].any()

print(len(anomaly_postings))
print(anomaly_postings.sum())


102429
850


In [15]:
from ai_ledger_guard.evaluation import *

precision, recall = precision_recall(weekend_postings, anomaly_postings)
print(round(precision, 4), round(recall, 4))


0.0206 0.38
